<a href="https://colab.research.google.com/github/shivamgiri007/AI_Repository/blob/main/sentiment%20analysis.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import time
import math
import torch
import torch.nn as nn
import torch.cuda.amp as amp
from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2ForSequenceClassification
from transformers.models.gpt2.modeling_gpt2 import Conv1D

In [2]:
train_path = "train.tsv"
val_path = "dev.tsv"

In [3]:
# Set model name
model_name = "gpt2"

# Load GPT-2 Tokenizer
tokenizer = GPT2Tokenizer.from_pretrained(model_name)

# Add a separate pad_token
if tokenizer.pad_token is None:
    # Use '<|PAD|>' as the padding token
    tokenizer.add_special_tokens({'pad_token': '<|PAD|>'})
    pad_token_id = tokenizer.pad_token_id
    print("Added new pad_token '<|PAD|>' with ID:", pad_token_id)

class SentimentDataset(Dataset):
    def __init__(self, file_path, tokenizer, max_length=None, pad_token_id=None):
        self.data = pd.read_csv(file_path, sep="\t")

        # Verify necessary columns in the CSV file
        required_columns = ["sentence", "label"]
        if not all(col in self.data.columns for col in required_columns):
            raise ValueError(f"CSV file must contain the following columns: {required_columns}")

        # Ensure labels are of integer type
        self.data["label"] = self.data["label"].astype(int)

        self.texts = self.data["sentence"].tolist()
        self.labels = self.data["label"].tolist()

        # Set pad_token_id, if not specified, use tokenizer's pad_token_id
        self.pad_token_id = pad_token_id if pad_token_id is not None else tokenizer.pad_token_id

        # Encode texts
        self.encoded_texts = []
        for text in self.texts:
            try:
                encoded = tokenizer.encode(text, add_special_tokens=True)
                self.encoded_texts.append(encoded)
            except Exception as e:
                raise ValueError(f"Error encoding text: {text[:50]}...") from e

        # Dynamically calculate max_length, or use specified max_length
        if max_length is None:
            self.max_length = self._longest_encoded_length()
        else:
            self.max_length = max_length
            # Truncate sequences longer than max_length
            self.encoded_texts = [
                encoded_text[:self.max_length] for encoded_text in self.encoded_texts
            ]

        # Pad all sequences and generate attention_mask
        self.padded_texts = []
        self.attention_masks = []
        for enc in self.encoded_texts:
            enc = enc[:self.max_length]
            attention_mask = [1] * len(enc)

            pad_len = self.max_length - len(enc)
            if pad_len > 0:
                enc += [self.pad_token_id] * pad_len
                attention_mask += [0] * pad_len

            self.padded_texts.append(enc)
            self.attention_masks.append(attention_mask)

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        input_ids = torch.tensor(self.padded_texts[idx], dtype=torch.long)
        attention_mask = torch.tensor(self.attention_masks[idx], dtype=torch.long)
        label = torch.tensor(self.labels[idx], dtype=torch.long)
        text = self.texts[idx]
        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "labels": label,
            "text": text
        }

    def _longest_encoded_length(self):
        return max(len(encoded_text) for encoded_text in self.encoded_texts)

# Create datasets
train_dataset = SentimentDataset(train_path, tokenizer, pad_token_id=pad_token_id)
val_dataset = SentimentDataset(val_path, tokenizer, max_length=train_dataset.max_length, pad_token_id=pad_token_id)
# test_dataset = SentimentDataset(test_path, tokenizer, max_length=train_dataset.max_length, pad_token_id=pad_token_id)

# Set DataLoader parameters
batch_size = 8
num_workers = 0

# Create DataLoaders
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=num_workers, drop_last=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)
# test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=num_workers, drop_last=False)

print(f"Number of training batches: {len(train_loader)}, Number of validation batches: {len(val_loader)}")

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

Added new pad_token '<|PAD|>' with ID: 50257
Number of training batches: 8418, Number of validation batches: 109


In [4]:
if torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")
print(f"Using device: {device}")

Using device: cuda


In [5]:
model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    pad_token_id=tokenizer.pad_token_id
)

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


# Only head

In [6]:
def classify_text(text, model, tokenizer, device, max_length, pad_token_id=50256):
    model = model.to(device)
    model.eval()
    enc = tokenizer.encode(text, add_special_tokens=True, truncation=True, max_length=max_length)
    att_mask = [1]*len(enc)
    pad_len = max_length - len(enc)
    if pad_len > 0:
        enc += [pad_token_id]*pad_len
        att_mask += [0]*pad_len

    input_ids = torch.tensor([enc], dtype=torch.long).to(device)
    attention_mask = torch.tensor([att_mask], dtype=torch.long).to(device)

    with torch.no_grad():
        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits
        predicted = torch.argmax(logits, dim=-1).item()
    return "positive" if predicted==1 else "negative"

sample_text_pos = "that loves its characters and communicates something rather beautiful about human nature"
sample_text_neg = "contains no wit , only labored gags"

print("(Before fine-tuning) Initial prediction of the classification head:")
print(f"positive sample => Prediction: {classify_text(sample_text_pos, model, tokenizer, device, train_dataset.max_length)}")
print(f"negative sample => Prediction: {classify_text(sample_text_neg, model, tokenizer, device, train_dataset.max_length)}")

(Before fine-tuning) Initial prediction of the classification head:
positive sample => Prediction: positive
negative sample => Prediction: positive


In [7]:
# add padding token
model.resize_token_embeddings(len(tokenizer))
model.to(device)

i = 0
k = 0
for param in model.base_model.parameters():
    param.requires_grad = False
    i+=param.numel()

print(f"Number of base parameters: {i}")


for param in model.score.parameters():
    param.requires_grad = True
    k+=param.numel()

print(f"\nTraining only the classification head, trainable parameters: {k}")

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


Number of base parameters: 124440576

Training only the classification head, trainable parameters: 1536


In [8]:
def eval_accuracy(model, loader, device):
    model.to(device)
    model.eval()

    total_loss = 0.0
    total_correct = 0
    total = 0
    with torch.no_grad():
        for batch_idx, batch in enumerate(loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

            loss = outputs.loss

            total_loss += loss.item()
            total_correct += (torch.argmax(outputs.logits, dim=1) == labels).sum().item()
            total += len(labels)

    avg_loss = total_loss / len(loader)
    accuracy = total_correct / total

    return avg_loss, accuracy



def train_head_only(model, train_loader, val_loader, device, epochs=3, lr=3e-5):
    optimizer = torch.optim.AdamW(model.score.parameters(), lr=lr)


    for epoch in range(epochs):
        model.train()
        total_loss = 0.0
        total_correct = 0
        total = 0
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids, attention_mask=attention_mask, labels=labels)

            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_loss += loss.item()
            total_correct += (torch.argmax(outputs.logits, dim=1) == labels).sum().item()
            total += len(labels)

            if (batch_idx+1) % 200 == 0:
                print(f"Epoch {epoch+1}, step {batch_idx+1}, loss = {loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        epoch_acc = total_correct / total
        print(f"Epoch {epoch+1}/{epochs}, Average training loss: {avg_loss:.4f}, accuracy: {epoch_acc*100:.2f}%")

        val_loss, val_acc = eval_accuracy(model, val_loader, device)
        print(f"Validation loss: {val_loss:.4f}, accuracy: {val_acc*100:.2f}%")

    return model


start_time_head = time.time()
model_finetuned_head = train_head_only(model, train_loader, val_loader, device, epochs=3, lr=3e-5)
end_time_head = time.time()

Epoch 1, step 200, loss = 0.9707
Epoch 1, step 400, loss = 1.2312
Epoch 1, step 600, loss = 0.9394
Epoch 1, step 800, loss = 0.7461
Epoch 1, step 1000, loss = 0.7486
Epoch 1, step 1200, loss = 0.6645
Epoch 1, step 1400, loss = 0.9929
Epoch 1, step 1600, loss = 0.5859
Epoch 1, step 1800, loss = 0.7219
Epoch 1, step 2000, loss = 0.8812
Epoch 1, step 2200, loss = 0.9131
Epoch 1, step 2400, loss = 0.6686
Epoch 1, step 2600, loss = 0.6476
Epoch 1, step 2800, loss = 0.6652
Epoch 1, step 3000, loss = 0.6537
Epoch 1, step 3200, loss = 0.6514
Epoch 1, step 3400, loss = 0.8096
Epoch 1, step 3600, loss = 0.6844
Epoch 1, step 3800, loss = 0.6940
Epoch 1, step 4000, loss = 0.6257
Epoch 1, step 4200, loss = 0.4912
Epoch 1, step 4400, loss = 0.6427
Epoch 1, step 4600, loss = 0.5960
Epoch 1, step 4800, loss = 0.5526
Epoch 1, step 5000, loss = 0.5431
Epoch 1, step 5200, loss = 0.7139
Epoch 1, step 5400, loss = 0.5586
Epoch 1, step 5600, loss = 0.7623
Epoch 1, step 5800, loss = 0.5188
Epoch 1, step 6000

In [9]:
_, train_accuracy_head = eval_accuracy(model_finetuned_head, train_loader, device)
_,val_accuracy_head = eval_accuracy(model_finetuned_head, val_loader, device)


finetune_head_time = (end_time_head - start_time_head) / 60

print(f"\n=== Fine-tuning only the classification head completed in {finetune_head_time:.2f} minutes ===")
print(f"Training accuracy: {train_accuracy_head*100:.2f}%")
print(f"Validation accuracy: {val_accuracy_head*100:.2f}%")
# print(f"Test accuracy: {test_accuracy_head*100:.2f}%")


=== Fine-tuning only the classification head completed in 14.48 minutes ===
Training accuracy: 78.70%
Validation accuracy: 79.36%


In [11]:
with open("sentiment_instances.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line)
        print(f"-> {classify_text(line, model_finetuned_head, tokenizer, device, train_dataset.max_length)}")
        print("----------------------\n")

A masterpiece four years in the making

-> negative
----------------------

Good fun , good action , good acting , good dialogue , good pace , good cinematography .

-> negative
----------------------

Ranks among Willams ' best screen work .

-> negative
----------------------

Yeah , these flicks are just that damn good .

-> negative
----------------------

The film is just a big , gorgeous , mind-blowing , breath-taking mess .

-> negative
----------------------

A story which fails to rise above its disgusting source material .

-> negative
----------------------

It 's not horrible , just horribly mediocre .

-> negative
----------------------

Stinks from start to finish , like a wet burlap sack of gloom .

-> negative
----------------------

`` The Adventures of Pluto Nash '' is a big time stinker .

-> negative
----------------------

Karmen moves like rhythm itself , her lips chanting to the beat , her long , braided hair doing little to wipe away the jeweled beads of sweat.


# LoRA

In [12]:
class LoRALayer(nn.Module):
    """
    Low-Rank Adaptation layer to inject trainable parameters A and B into original weight update.
    """
    def __init__(self, in_dim, out_dim, rank, alpha=1.0):
        super().__init__()
        self.rank = rank
        self.alpha = alpha

        # Low-rank matrices
        self.A = nn.Parameter(torch.empty(in_dim, rank))
        self.B = nn.Parameter(torch.empty(rank, out_dim))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
        nn.init.zeros_(self.B)

        # Explanation log: GPT-2 base has hidden_dim=768; rank=16, alpha=32 by default.
        print(f"[LoRALayer] in_dim={in_dim}, out_dim={out_dim}, rank={rank}, alpha={alpha}")

    def forward(self, x):
        # Decomposition: alpha * (x @ A @ B)
        return self.alpha * (x @ self.A @ self.B)

class LinearWithLoRA(nn.Module):
    """
    Wrapper for nn.Linear that adds a LoRA output to the original linear output.
    """
    def __init__(self, linear_module, rank, alpha=1.0):
        super().__init__()
        self.linear = linear_module
        self.lora   = LoRALayer(linear_module.in_features, linear_module.out_features, rank, alpha)

    def forward(self, x):
        return self.linear(x) + self.lora(x)

class Conv1DWithLoRA(nn.Module):
    """
    Wrapper for Conv1D that adds a LoRA output to the original Conv1D output.
    """
    def __init__(self, conv1d_module: Conv1D, rank, alpha=1.0):
        super().__init__()
        self.conv = conv1d_module
        in_dim, out_dim = conv1d_module.weight.shape
        self.lora = LoRALayer(in_dim, out_dim, rank, alpha)

    def forward(self, x):
        out_normal = self.conv(x)
        B, S, hidden_dim = x.shape
        x_2d = x.view(B*S, hidden_dim)
        out_lora_2d = self.lora(x_2d)
        out_lora_3d = out_lora_2d.view(B, S, -1)
        return out_normal + out_lora_3d

def replace_modules_with_lora(module, rank=16, alpha=32):
    """
    Recursively replace GPT-2 submodules (c_fc, c_proj) with LoRA wrappers.
    """
    for name, child in list(module.named_children()):
        if isinstance(child, nn.Linear) and name in ["c_fc", "c_proj"]:
            new_module = LinearWithLoRA(child, rank, alpha)
            setattr(module, name, new_module)
        elif isinstance(child, Conv1D) and name in ["c_fc", "c_proj"]:
            new_module = Conv1DWithLoRA(child, rank, alpha)
            setattr(module, name, new_module)
        else:
            replace_modules_with_lora(child, rank, alpha)

def freeze_original_parameters(model):
    """
    Freeze all parameters except LoRA layers and classifier.
    """
    for name, param in model.named_parameters():
        if "lora" not in name.lower() and "classifier" not in name.lower():
            param.requires_grad = False

def print_trainable_parameters(model):
    """
    Print the number of trainable parameters vs. total parameters.
    """
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    total     = sum(p.numel() for p in model.parameters())
    print(f"Trainable params: {trainable} / Total params: {total}")
    return trainable

def show_gradient_norms(model):
    """
    Print gradient norms for LoRA layers to confirm only LoRA + classifier receive gradients.
    """
    for name, param in model.named_parameters():
        if param.requires_grad and param.grad is not None:
            print(f"Gradient Norm for {name}: {param.grad.norm():.4f}")

In [13]:
model = GPT2ForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    pad_token_id=tokenizer.pad_token_id
)

model.resize_token_embeddings(len(tokenizer))

replace_modules_with_lora(model, rank=16, alpha=32)
model.to(device)

Some weights of GPT2ForSequenceClassification were not initialized from the model checkpoint at gpt2 and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


[LoRALayer] in_dim=768, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=3072, rank=16, alpha=32
[LoRALayer] in_dim=3072, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=3072, rank=16, alpha=32
[LoRALayer] in_dim=3072, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=3072, rank=16, alpha=32
[LoRALayer] in_dim=3072, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=3072, rank=16, alpha=32
[LoRALayer] in_dim=3072, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=3072, rank=16, alpha=32
[LoRALayer] in_dim=3072, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=768, rank=16, alpha=32
[LoRALayer] in_dim=768, out_dim=3072, rank=16, alpha=32
[LoRALayer] in_dim=3072, out_dim=768, rank=16, alpha=3

GPT2ForSequenceClassification(
  (transformer): GPT2Model(
    (wte): Embedding(50258, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1DWithLoRA(
            (conv): Conv1D(nf=768, nx=768)
            (lora): LoRALayer()
          )
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1DWithLoRA(
            (conv): Conv1D(nf=3072, nx=768)
            (lora): LoRALayer()
          )
          (c_proj): Conv1DWithLoRA(
            (conv): Conv1D(nf=768, nx=3072)
            (lora): LoRALayer()
          )
          (act): NewGELUActivation()
          (drop

In [14]:
def train_lora(model, train_loader, val_loader, device, epochs=3, lr=1e-4):
    freeze_original_parameters(model)
    print_trainable_parameters(model)

    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)
    # loss_fn = nn.CrossEntropyLoss()
    scaler = amp.GradScaler()

    # train_losses, val_accs = [], []
    # start_time = time.time()

    for epoch in range(epochs):
        model.train()
        total_loss = 0
        total_correct = 0
        total = 0
        for batch_idx, batch in enumerate(train_loader):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["labels"].to(device)

            optimizer.zero_grad()
            with amp.autocast():
                # outputss = forward_for_classification(model, input_ids, attention_mask, device)
                outputs = model(input_ids, attention_mask=attention_mask, labels=labels)
                loss = outputs.loss

            scaler.scale(loss).backward()

            scaler.step(optimizer)
            scaler.update()

            total_loss += loss.item()
            total_correct += (torch.argmax(outputs.logits, dim=1) == labels).sum().item()
            total += len(labels)

            if (batch_idx+1) % 200 == 0:
                print(f"Epoch {epoch+1}, step {batch_idx+1}, loss = {loss.item():.4f}")

        avg_loss = total_loss / len(train_loader)
        epoch_acc = total_correct / total
        print(f"Epoch {epoch+1}/{epochs}, Average training loss: {avg_loss:.4f}, accuracy: {epoch_acc*100:.2f}%")

        val_loss, val_acc = eval_accuracy(model, val_loader, device)
        print(f"Validation loss: {val_loss:.4f}, accuracy: {val_acc*100:.2f}%")

    return model

start_time_head = time.time()
model_lora = train_lora(model, train_loader, val_loader, device, epochs=3, lr=1e-4)
end_time_head = time.time()

Trainable params: 1769472 / Total params: 126211584


<ipython-input-14-be4ddd1bf356>:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = amp.GradScaler()
<ipython-input-14-be4ddd1bf356>:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with amp.autocast():


Epoch 1, step 200, loss = 0.5202
Epoch 1, step 400, loss = 0.6139
Epoch 1, step 600, loss = 1.0348
Epoch 1, step 800, loss = 0.7285
Epoch 1, step 1000, loss = 0.7102
Epoch 1, step 1200, loss = 0.7141
Epoch 1, step 1400, loss = 0.6195
Epoch 1, step 1600, loss = 0.6110
Epoch 1, step 1800, loss = 0.6041
Epoch 1, step 2000, loss = 0.6516
Epoch 1, step 2200, loss = 0.7601
Epoch 1, step 2400, loss = 0.5487
Epoch 1, step 2600, loss = 0.8417
Epoch 1, step 2800, loss = 0.8638
Epoch 1, step 3000, loss = 0.6823
Epoch 1, step 3200, loss = 0.6198
Epoch 1, step 3400, loss = 0.4475
Epoch 1, step 3600, loss = 0.7079
Epoch 1, step 3800, loss = 0.3706
Epoch 1, step 4000, loss = 0.5253
Epoch 1, step 4200, loss = 0.6880
Epoch 1, step 4400, loss = 0.8299
Epoch 1, step 4600, loss = 0.7091
Epoch 1, step 4800, loss = 0.7090
Epoch 1, step 5000, loss = 0.3756
Epoch 1, step 5200, loss = 0.5787
Epoch 1, step 5400, loss = 0.4455
Epoch 1, step 5600, loss = 0.2912
Epoch 1, step 5800, loss = 1.1126
Epoch 1, step 6000

In [15]:
def save_lora_params(model, save_path="lora_params.pt"):
    """
    Save only LoRA-related parameters (and the classifier) for demonstration.
    """
    lora_dict = {
        k: v for k, v in model.state_dict().items()
        if "lora" in k.lower() or "classifier" in k.lower()
    }
    torch.save(lora_dict, save_path)
    print(f"LoRA params saved to {save_path}")

def load_lora_params(model, load_path="lora_params.pt"):
    """
    Load LoRA parameters into a GPT-2 model that already has LoRA layers.
    """
    loaded_dict = torch.load(load_path, map_location=device)
    model.load_state_dict(loaded_dict, strict=False)
    print(f"LoRA params loaded from {load_path}")

save_lora_params(model_lora)

LoRA params saved to lora_params.pt


In [16]:
_, train_accuracy_lora = eval_accuracy(model_lora, train_loader, device)
_,val_accuracy_lora = eval_accuracy(model_lora, val_loader, device)


lora_time = (end_time_head - start_time_head) / 60

print(f"\n=== LORA fine-tuning completed in {lora_time:.2f} minutes ===")
print(f"Training accuracy: {train_accuracy_lora*100:.2f}%")
print(f"Validation accuracy: {val_accuracy_lora*100:.2f}%")


=== LORA fine-tuning completed in 20.21 minutes ===
Training accuracy: 83.25%
Validation accuracy: 80.16%


In [18]:
with open("sentiment_instances.txt", "r", encoding="utf-8") as f:
    for line in f:
        print(line)
        print(f"-> {classify_text(line, model_lora, tokenizer, device, train_dataset.max_length)}")
        print("----------------------\n")

A masterpiece four years in the making

-> positive
----------------------

Good fun , good action , good acting , good dialogue , good pace , good cinematography .

-> positive
----------------------

Ranks among Willams ' best screen work .

-> positive
----------------------

Yeah , these flicks are just that damn good .

-> positive
----------------------

The film is just a big , gorgeous , mind-blowing , breath-taking mess .

-> positive
----------------------

A story which fails to rise above its disgusting source material .

-> negative
----------------------

It 's not horrible , just horribly mediocre .

-> negative
----------------------

Stinks from start to finish , like a wet burlap sack of gloom .

-> negative
----------------------

`` The Adventures of Pluto Nash '' is a big time stinker .

-> positive
----------------------

Karmen moves like rhythm itself , her lips chanting to the beat , her long , braided hair doing little to wipe away the jeweled beads of sweat.
